In [22]:
# %%
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
# import time as py_time # No longer used directly, can be removed

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, KFold

# Suppress TensorFlow logging (1 = INFO, 2 = WARNING, 3 = ERROR)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.get_logger().setLevel('ERROR')  # Suppress Keras/TF Python-level warnings/info

from tinkoff.invest import Client, CandleInterval, RequestError
from tinkoff.invest.utils import now, quotation_to_decimal

from config import TOKEN  # Assuming config.py is in the same directory or Python path

# Multiprocessing imports
import multiprocessing

# import pickle # Not explicitly needed if Keras models pickle fine

TINKOFF_API_TOKEN = TOKEN

In [23]:
# --- Script Configuration ---
DAYS_HISTORY = 10
CANDLE_TIMEFRAME = CandleInterval.CANDLE_INTERVAL_1_MIN
N_STEPS_IN = 150  # Input sequence length
N_STEPS_OUT = 50  # Output sequence length (predict N_STEPS_OUT future points)
FEATURE_COLUMN = 'close'
FUTURE_MINUTES = 120  # Number of future minutes to predict step-by-step

# --- Cross-validation and Plotting Configuration ---
N_CV_SPLITS = 4  # Number of folds for cross-validation
PLOT_FOLD_PREDICTIONS = False  # Set to True to see plots from each CV fold (non-blocking)
PLOT_FINAL_TEST_PREDICTIONS = True  # Set to True to see final test plot
PLOT_FUTURE_FORECAST = True  # Set to True to see future forecast plot


In [24]:
def configure_tensorflow_memory():
    """Configures TensorFlow to use GPU memory growth to avoid errors in multiprocessing."""
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            # print("TensorFlow GPU memory growth configured.")
        except RuntimeError as e:
            # Memory growth must be set before GPUs have been initialized
            print(f"TensorFlow GPU memory growth configuration error: {e}", file=sys.stderr)


# Configure TF memory globally at the start of the script (for the main process)
configure_tensorflow_memory()


In [25]:
def get_tinkoff_candles(token, figi, days, interval):
    print(f"Fetching candle data for FIGI: {figi} for the last {days} days with interval: {interval.name}")
    try:
        with Client(token) as client:
            end_dt = now()
            start_dt = end_dt - timedelta(days=days)
            candles = list(client.get_all_candles(figi=figi, from_=start_dt, to=end_dt, interval=interval))

            if not candles:
                print(f"No candle data returned for FIGI: {figi}", file=sys.stderr)
                return pd.DataFrame()

            df = pd.DataFrame([{
                'time': pd.Timestamp(c.time).tz_localize(None),  # Ensure tz-naive for consistency
                'open': float(quotation_to_decimal(c.open)),
                'high': float(quotation_to_decimal(c.high)),
                'low': float(quotation_to_decimal(c.low)),
                'close': float(quotation_to_decimal(c.close)),
                'volume': c.volume
            } for c in candles])

            df = df.set_index('time').sort_index().dropna()
            print(f"Fetched {len(df)} candles. Data range: {df.index.min()} to {df.index.max()}")
            # print("Sample of fetched data (first 3 and last 3 rows):")
            # print(pd.concat([df.head(3), df.tail(3)]))
            return df
    except RequestError as e:
        print(f"Tinkoff API RequestError while fetching candles for {figi}: {e}", file=sys.stderr)
        return pd.DataFrame()
    except Exception as e:
        print(f"An unexpected error occurred while fetching candles for {figi}: {e}", file=sys.stderr)
        return pd.DataFrame()


In [26]:
def prepare_data(df, feature_col):
    if feature_col not in df.columns:
        print(f"Feature column '{feature_col}' not found in DataFrame.", file=sys.stderr)
        return None, None
    values = df[[feature_col]].dropna().values.astype(float)
    if values.shape[0] == 0:
        print(f"No data available in feature column '{feature_col}' after dropna.", file=sys.stderr)
        return None, None
    print(f"Prepared data shape for scaling: {values.shape}")
    return values, df.index  # Return original timestamps as well


In [27]:
def split_sequences(data, n_steps_in, n_steps_out, timestamps):
    X, y, times_y = [], [], []
    if len(data) < n_steps_in + n_steps_out:
        print(f"Not enough data to create sequences. Data length: {len(data)}, required: {n_steps_in + n_steps_out}",
              file=sys.stderr)
        return np.array(X), np.array(y), np.array(times_y)

    for i in range(len(data) - n_steps_in - n_steps_out + 1):
        X.append(data[i: i + n_steps_in])
        y.append(data[i + n_steps_in: i + n_steps_in + n_steps_out])
        times_y.append(timestamps[i + n_steps_in])  # Timestamp of the first y value

    X_np, y_np, times_y_np = np.array(X), np.array(y), np.array(times_y)
    print(f"Split data into X shape: {X_np.shape}, y shape: {y_np.shape}, times_y shape: {times_y_np.shape}")
    return X_np, y_np, times_y_np


In [28]:
def build_lstm_model(n_steps_in, n_steps_out):
    model = Sequential([
        LSTM(75, return_sequences=True, input_shape=(n_steps_in, 1)),
        Dropout(0.2),
        LSTM(75),
        Dropout(0.2),
        Dense(50, activation='relu'),
        Dense(n_steps_out)  # Linear activation for regression
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model


In [29]:
def plot_predictions(dates, actual, predicted, title, filename_prefix=None):
    plt.figure(figsize=(14, 7))
    plt.plot(dates, actual, label='Actual', marker='.', linestyle='-')
    plt.plot(dates, predicted, label='Predicted', marker='.', linestyle='--')
    plt.title(title)
    plt.xlabel("Date and Time")
    plt.ylabel("Price")
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    if filename_prefix:
        plt.savefig(f"{filename_prefix}_{title.replace(' ', '_').lower()}.png")
        print(f"Plot saved: {filename_prefix}_{title.replace(' ', '_').lower()}.png")
        plt.close()  # Close the figure after saving to free memory
    else:
        plt.show(block=False)  # Non-blocking display
        plt.pause(0.1)  # Allow GUI time to render


In [30]:
def train_and_evaluate_fold(args):
    fold_num, train_idx, val_idx, X_data, y_data, scaler, n_steps_in, n_steps_out, plot_flag, figi_for_plot_title = args

    # Configure TF memory growth for this worker process
    configure_tensorflow_memory()

    print(f"--- Training model for fold {fold_num + 1}/{N_CV_SPLITS} (Process ID: {os.getpid()}) ---")

    model = build_lstm_model(n_steps_in, n_steps_out)

    X_train_fold, y_train_fold = X_data[train_idx], y_data[train_idx]
    X_val_fold, y_val_fold = X_data[val_idx], y_data[val_idx]

    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    history = model.fit(X_train_fold, y_train_fold,
                        validation_data=(X_val_fold, y_val_fold),
                        epochs=50,  # Max epochs
                        batch_size=32,
                        verbose=0,  # Reduce verbosity in parallel processes
                        callbacks=[early_stopping])

    # Predictions are (num_samples, n_steps_out)
    preds_scaled = model.predict(X_val_fold, verbose=0)

    # For MSE and plotting, typically use the first step of the multi-step-ahead prediction
    # Or evaluate across all N_STEPS_OUT if that's the goal. Here, let's focus on the first.
    # y_val_fold shape: (n_samples, N_STEPS_OUT, 1 feature) -> squeeze to (n_samples, N_STEPS_OUT)
    # preds_scaled shape: (n_samples, N_STEPS_OUT)

    # Inverse transform requires 2D array: (n_samples, n_features)
    # We scale only one feature (close price)
    # To inverse transform each step in N_STEPS_OUT, we'd loop or reshape carefully.
    # For simplicity, using first step for MSE reporting as in original code.

    y_val_first_step_scaled = y_val_fold[:, 0].reshape(-1, 1)
    preds_first_step_scaled = preds_scaled[:, 0].reshape(-1, 1)

    y_val_first_step_inv = scaler.inverse_transform(y_val_first_step_scaled)
    preds_first_step_inv = scaler.inverse_transform(preds_first_step_scaled)

    mse = mean_squared_error(y_val_first_step_inv, preds_first_step_inv)
    print(f"Fold {fold_num + 1} MSE (1st step): {mse:.6f}. Min val_loss: {min(history.history['val_loss']):.6f}")

    if plot_flag:
        # For plotting, generate dummy time indices or pass actual validation timestamps if needed
        val_time_indexes = pd.date_range(start=pd.Timestamp.now(), periods=len(y_val_first_step_inv),
                                         freq='T')  # Dummy times
        plot_predictions(val_time_indexes,
                         y_val_first_step_inv.flatten(),
                         preds_first_step_inv.flatten(),
                         title=f"{figi_for_plot_title} CV Fold {fold_num + 1} Prediction")
        # filename_prefix=f"{figi_for_plot_title}_fold_{fold_num+1}" # Optionally save plots

    # Return the trained model, its MSE, and history
    # Keras models are generally picklable if they don't use complex custom components.
    return model, mse, history



In [31]:
def cross_validate_lstm(X_trainval, y_trainval, scaler, n_splits, figi_for_plot_title):
    best_mse = float('inf')
    best_model_from_fold = None  # Model from the best fold (trained on a subset of X_trainval)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_args_list = []
    for fold_num, (train_idx, val_idx) in enumerate(kf.split(X_trainval)):
        fold_args_list.append((
            fold_num, train_idx, val_idx,
            X_trainval, y_trainval, scaler,
            N_STEPS_IN, N_STEPS_OUT, PLOT_FOLD_PREDICTIONS, figi_for_plot_title
        ))

    num_processes = min(os.cpu_count() or 1, n_splits)
    print(f"\nStarting cross-validation with {num_processes} processes for {n_splits} folds...")

    results = []
    pool = None
    try:
        # 'spawn' start method is generally safer for TensorFlow/Keras with multiprocessing
        # On Windows it's the default. On Linux/macOS, 'fork' is default but can cause issues.
        # ctx = multiprocessing.get_context('spawn') # Force spawn context
        # pool = ctx.Pool(processes=num_processes)
        pool = multiprocessing.Pool(processes=num_processes)  # Using default context
        results = pool.map(train_and_evaluate_fold, fold_args_list)
    except Exception as e:
        print(f"Error during multiprocessing pool execution: {e}", file=sys.stderr)
        print("Consider falling back to sequential execution or debugging the worker function.", file=sys.stderr)
        # Fallback (optional, for robustness or debugging):
        # print("Falling back to sequential cross-validation due to pool error.")
        # results = [train_and_evaluate_fold(args) for args in fold_args_list]
        raise  # Re-raise the error to stop the script if CV fails critically
    finally:
        if pool:
            pool.close()
            pool.join()

    for model_fold, mse_fold, _ in results:  # history_fold is also available
        if mse_fold < best_mse:
            best_mse = mse_fold
            best_model_from_fold = model_fold  # This is the model object from the best fold

    print(f"\nCross-validation finished. Best fold validation MSE (1st step): {best_mse:.6f}")

    if best_model_from_fold is None:
        print("No model was successfully trained during cross-validation.", file=sys.stderr)
        return None

    # Standard practice: Retrain the model on the *entire* X_trainval dataset
    # using the (hyper)parameters of the best fold (here, architecture is fixed).
    print("\nRetraining the model architecture on the full training+validation set (X_trainval)...")
    final_model_for_testing = build_lstm_model(N_STEPS_IN, N_STEPS_OUT)

    early_stopping_final = EarlyStopping(monitor='loss', patience=10,
                                         restore_best_weights=True)  # Monitor training loss
    # Note: No validation_data here, as we are training on the whole X_trainval.
    # One could split X_trainval again for a final validation if desired, but often not done.
    history_final_model = final_model_for_testing.fit(X_trainval, y_trainval,
                                                      epochs=100,  # Potentially more epochs for final model
                                                      batch_size=32,
                                                      verbose=1,
                                                      callbacks=[early_stopping_final])
    print(f"Final model retrained on X_trainval. Final training loss: {min(history_final_model.history['loss']):.6f}")
    return final_model_for_testing


In [32]:
def final_test(model, X_test, y_test_scaled, times_y_test, scaler, figi_for_plot_title):
    print("\n--- Final Model Testing ---")
    if model is None:
        print("No model available for final testing.", file=sys.stderr)
        return

    preds_scaled = model.predict(X_test, verbose=0)  # (num_samples, N_STEPS_OUT)

    # We need to compare apples to apples. If y_test_scaled is (num_samples, N_STEPS_OUT, 1)
    # and preds_scaled is (num_samples, N_STEPS_OUT), adjust for scaler.

    # For detailed evaluation, one might evaluate each of the N_STEPS_OUT predictions.
    # For this example, let's use the first predicted step, similar to CV.
    y_test_first_step_scaled = y_test_scaled[:, 0].reshape(-1, 1)
    preds_first_step_scaled = preds_scaled[:, 0].reshape(-1, 1)

    y_test_first_step_inv = scaler.inverse_transform(y_test_first_step_scaled)
    preds_first_step_inv = scaler.inverse_transform(preds_first_step_scaled)

    mse = mean_squared_error(y_test_first_step_inv, preds_first_step_inv)
    print(f"Test MSE (1st step prediction): {mse:.6f}")

    result_df = pd.DataFrame({
        'actual_close': y_test_first_step_inv.flatten(),
        'predicted_close': preds_first_step_inv.flatten()
    }, index=pd.to_datetime(times_y_test))  # Use the timestamps corresponding to y_test

    result_df = result_df.sort_index()

    print("\nSample of Test Predictions (Actual vs. Predicted for 1st step):")
    print(pd.concat([result_df.head(3), result_df.tail(3)]))

    if PLOT_FINAL_TEST_PREDICTIONS:
        plot_predictions(result_df.index,
                         result_df['actual_close'],
                         result_df['predicted_close'],
                         title=f"{figi_for_plot_title} Final Test: Actual vs Predicted (1st step)")
        # filename_prefix=f"{figi_for_plot_title}_final_test" # Optionally save



In [33]:
if len(sys.argv) < 2:
    print("Usage: python test_validation.py <FIGI> [--no-plots]", file=sys.stderr)
    sys.exit(1)

INSTRUMENT_FIGI = "BBG004730N88"
print(f"=== Starting Model Training and Prediction for FIGI: {INSTRUMENT_FIGI} ===")

# Handle optional plotting flags from CLI (simple version)
if "--no-plots" in sys.argv:
    print("Plotting disabled via command line argument.")
    PLOT_FOLD_PREDICTIONS = False
    PLOT_FINAL_TEST_PREDICTIONS = False
    PLOT_FUTURE_FORECAST = False
if "--no-cv-plots" in sys.argv: PLOT_FOLD_PREDICTIONS = False
if "--no-test-plots" in sys.argv: PLOT_FINAL_TEST_PREDICTIONS = False
if "--no-future-plots" in sys.argv: PLOT_FUTURE_FORECAST = False

# 1. Get Data
df_raw = get_tinkoff_candles(TINKOFF_API_TOKEN, INSTRUMENT_FIGI, DAYS_HISTORY, CANDLE_TIMEFRAME)
if df_raw.empty:
    print(f"Failed to fetch or process data for {INSTRUMENT_FIGI}. Exiting.", file=sys.stderr)
    sys.exit(1)

=== Starting Model Training and Prediction for FIGI: BBG004730N88 ===
Fetching candle data for FIGI: BBG004730N88 for the last 10 days with interval: CANDLE_INTERVAL_1_MIN
Fetched 10253 candles. Data range: 2025-05-02 03:59:00 to 2025-05-11 20:49:00


In [35]:
# 2. Prepare Data
price_values, original_timestamps = prepare_data(df_raw, FEATURE_COLUMN)
if price_values is None or price_values.shape[0] < (N_STEPS_IN + N_STEPS_OUT):  # Ensure enough data after prep
    print(f"Not enough data after preparation for {INSTRUMENT_FIGI}. " \
          f"Required: {N_STEPS_IN + N_STEPS_OUT}, Got: {price_values.shape[0] if price_values is not None else 0}. Exiting.",
          file=sys.stderr)
    sys.exit(1)

scaler = MinMaxScaler()
scaled_prices = scaler.fit_transform(price_values)


Prepared data shape for scaling: (10253, 1)


In [36]:
# 3. Split into sequences
# original_timestamps correspond to each row in price_values / scaled_prices
X_all, y_all_scaled, times_y_all = split_sequences(scaled_prices, N_STEPS_IN, N_STEPS_OUT, original_timestamps)
# X_all shape: (samples, N_STEPS_IN, 1 feature)
# y_all_scaled shape: (samples, N_STEPS_OUT, 1 feature)
# times_y_all shape: (samples,) - timestamps for the first element of each y sequence
# После получения price_values:

X_trainval, X_test = train_test_split(price_values, test_size=0.2, shuffle=False)
scaler = MinMaxScaler()
X_trainval_scaled = scaler.fit_transform(X_trainval)
X_test_scaled = scaler.transform(X_test)

if X_all.shape[0] == 0:
    print(f"Not enough data to form sequences for {INSTRUMENT_FIGI}. Exiting.", file=sys.stderr)
    sys.exit(1)

# Reshape X for LSTM: (samples, timesteps, features)
# Already (samples, N_STEPS_IN, 1) if scaled_prices was (N,1)
# If split_sequences returns X with shape (samples, N_STEPS_IN) for single feature, then reshape:
if X_all.ndim == 2:  # If X is (samples, N_STEPS_IN)
    X_all = X_all.reshape((X_all.shape[0], X_all.shape[1], 1))


Split data into X shape: (10054, 150, 1), y shape: (10054, 50, 1), times_y shape: (10054,)


In [37]:
# 4. Train/Test Split
# Ensure y_all_scaled and times_y_all are split consistently with X_all
X_trainval, X_test, y_trainval_scaled, y_test_scaled, times_y_trainval, times_y_test = train_test_split(
    X_all, y_all_scaled, times_y_all, test_size=0.2, shuffle=True, random_state=42
)
print(f"\nData Split: X_trainval: {X_trainval.shape}, y_trainval_scaled: {y_trainval_scaled.shape}, " \
      f"X_test: {X_test.shape}, y_test_scaled: {y_test_scaled.shape}")
print(f"Timestamps: times_y_trainval: {len(times_y_trainval)}, times_y_test: {len(times_y_test)}")



Data Split: X_trainval: (8043, 150, 1), y_trainval_scaled: (8043, 50, 1), X_test: (2011, 150, 1), y_test_scaled: (2011, 50, 1)
Timestamps: times_y_trainval: 8043, times_y_test: 2011


In [39]:
# 5. Cross-Validation and Model Training
# The model returned here is trained on the full X_trainval, y_trainval_scaled
trained_model = cross_validate_lstm(X_trainval, y_trainval_scaled, scaler, n_splits=N_CV_SPLITS,
                                    figi_for_plot_title=INSTRUMENT_FIGI)

if trained_model is None:
    print(f"Model training failed for {INSTRUMENT_FIGI}. Exiting.", file=sys.stderr)
    sys.exit(1)


Starting cross-validation with 4 processes for 4 folds...
--- Training model for fold 1/4 (Process ID: 64289) ---


/home/marcus2tut/.local/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


--- Training model for fold 2/4 (Process ID: 64290) ---
--- Training model for fold 3/4 (Process ID: 64291) ---


/home/marcus2tut/.local/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


--- Training model for fold 4/4 (Process ID: 64292) ---


/home/marcus2tut/.local/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/home/marcus2tut/.local/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
Process ForkPoolWorker-2:
Process ForkPoolWorker-1:
Process ForkPoolWorker-3:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.11

KeyboardInterrupt: 

In [ ]:
# 6. Final Test
final_test(trained_model, X_test, y_test_scaled, times_y_test, scaler, figi_for_plot_title=INSTRUMENT_FIGI)


In [ ]:
# 7. Future Price Prediction
print(f"\n=== Future Prediction for {INSTRUMENT_FIGI} (next {FUTURE_MINUTES} minutes) ===")

# Take the last N_STEPS_IN from the *entire* scaled dataset to predict from the most recent data
last_sequence_scaled = scaled_prices[-N_STEPS_IN:].reshape(1, N_STEPS_IN, 1)

future_predictions_scaled_steps = []  # Store individual predicted steps
current_input_sequence = last_sequence_scaled.copy()


In [ ]:
for i in range(FUTURE_MINUTES):
    # Predict N_STEPS_OUT ahead, but only use the first step for iterative prediction
    # model.predict output shape: (1, N_STEPS_OUT)
    predicted_all_steps_ahead_scaled = trained_model.predict(current_input_sequence, verbose=0)[
        0]  # Get (N_STEPS_OUT,)

    # Use the first step of the prediction
    next_step_scaled = predicted_all_steps_ahead_scaled[0]
    future_predictions_scaled_steps.append(next_step_scaled)

    # Update the input sequence: remove oldest, append newest prediction
    # current_input_sequence is (1, N_STEPS_IN, 1)
    # new_step_for_sequence needs to be (1,1) then added to (N_STEPS_IN-1, 1)
    new_step_reshaped = np.array([[next_step_scaled]])  # shape (1,1)
    current_input_sequence = np.append(current_input_sequence[0, 1:, 0], new_step_reshaped.flatten()).reshape(1,
                                                                                                              N_STEPS_IN,
                                                                                                              1)


In [ ]:
future_predictions_scaled_array = np.array(future_predictions_scaled_steps).reshape(-1, 1)
future_predictions_unscaled = scaler.inverse_transform(future_predictions_scaled_array)

last_known_time = df_raw.index[-1]
future_times = pd.date_range(start=last_known_time + pd.Timedelta(minutes=1),
                             periods=FUTURE_MINUTES, freq='T')  # Assuming 1-minute interval

future_df = pd.DataFrame({
    'predicted_close': future_predictions_unscaled.flatten()
}, index=future_times)
future_df.index.name = 'datetime'

print("\n--- Predicted Future Values ---")
if not future_df.empty:
    # Print first and last few predicted values
    print(pd.concat([future_df.head(3), future_df.tail(3)]))
else:
    print("No future data predicted.")
print("-----------------------------\n")

if PLOT_FUTURE_FORECAST and not future_df.empty:
    plt.figure(figsize=(12, 6))
    plt.plot(future_df.index, future_df['predicted_close'], label='Future Predicted Price', color='red',
             linestyle='--')
    # Optionally, plot some historical data for context
    historical_context_length = N_STEPS_IN  # Show last N_STEPS_IN actual prices
    if len(df_raw[FEATURE_COLUMN]) >= historical_context_length:
        plt.plot(df_raw.index[-historical_context_length:],
                 df_raw[FEATURE_COLUMN].values[-historical_context_length:],
                 label='Historical Price', color='blue')
    plt.title(f"{INSTRUMENT_FIGI} Price Forecast for next {FUTURE_MINUTES} minutes")
    plt.xlabel("Date and Time")
    plt.ylabel("Price")
    plt.xticks(rotation=45)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show(block=False)
    plt.pause(0.1)
    # plt.savefig(f"{INSTRUMENT_FIGI}_future_forecast.png") # Optionally save
    # plt.close()

In [ ]:
# 8. Save forecast to CSV for main.py
output_csv_path = "forecast_result.csv"
try:
    future_df.to_csv(output_csv_path)
    print(f"Future forecast results saved to: {output_csv_path} for FIGI: {INSTRUMENT_FIGI}")
except Exception as e:
    print(f"Error saving forecast results to CSV {output_csv_path}: {e}", file=sys.stderr)
    sys.exit(1)

print(f"=== Prediction script for FIGI: {INSTRUMENT_FIGI} finished successfully. ===")

# If plots were non-blocking, script might exit before they are fully viewed.
# Add a small pause or input() if running interactively and want to see plots.
if (PLOT_FOLD_PREDICTIONS or PLOT_FINAL_TEST_PREDICTIONS or PLOT_FUTURE_FORECAST) and \
        "--no-plots" not in sys.argv and \
        not any(arg in sys.argv for arg in ["--no-cv-plots", "--no-test-plots", "--no-future-plots"]):
    try:
        # Keep plots open for a bit if shown non-blockingly
        print("\nDisplaying plots. Close plot windows or press Ctrl+C in terminal to exit if plots block.")
        plt.show()  # This will block until all figures are closed.
    except Exception as e:
        print(f"Final plt.show() error: {e}. Plots might have already closed or not shown.", file=sys.stderr)